# Six labels, a thousand moths

A museum drawer holds 1,000 moth specimens of two species, each measured twice, and an expert has identified six of them, three of each species. Here you label the rest three ways: by the nearest identified specimen, by self-training that absorbs its most confident guesses one round at a time, and by label propagation over a similarity graph. The true species of every moth is in the notebook only to score each method, and no method sees it.

**How to work through it.** Each part asks for one function or class. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your code is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first. It builds the measurements with a seeded random generator, picks the six identified moths, and defines `accuracy`, which scores labels against the hidden species on the unlabeled moths, and `show`, which draws the drawer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

# 1,000 moths of two species, as (forewing length, antenna length) after
# scaling. The two species trace two interleaved crescents in this plane.
rng = np.random.default_rng(0)
n_each = 500
t = rng.uniform(0, np.pi, n_each)
species_0 = np.column_stack([np.cos(t), np.sin(t)])               # the upper crescent
species_1 = np.column_stack([1 - np.cos(t), 0.5 - np.sin(t)])     # the lower one, shifted
X = np.vstack([species_0, species_1]) + rng.normal(0, 0.08, (2 * n_each, 2))
y_true = np.repeat([0, 1], n_each)                                # hidden: only the scoring sees it

# The expert identified three moths of each species. Everything else is unlabeled.
labeled = np.concatenate([rng.choice(np.flatnonzero(y_true == c), 3, replace=False) for c in (0, 1)])
y_labeled = y_true[labeled]
unlabeled = np.setdiff1d(np.arange(len(X)), labeled)
print("moths:", len(X), "  identified:", len(labeled), "  unlabeled:", len(unlabeled))


def accuracy(pred):
    """Fraction of the unlabeled moths whose label matches the hidden species."""
    return float(np.mean(pred[unlabeled] == y_true[unlabeled]))


def show(ax, labels, title):
    """The drawer, colored by `labels`, with the six identified moths as stars."""
    ax.scatter(X[:, 0], X[:, 1], c=labels, cmap="coolwarm", s=6, vmin=0, vmax=1)
    ax.scatter(X[labeled, 0], X[labeled, 1], c=y_labeled, cmap="coolwarm", marker="*", s=200, edgecolor="black", vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])


fig, ax = plt.subplots(figsize=(5, 3.5))
show(ax, np.full(len(X), 0.5), "six identified moths, 994 unlabeled")
plt.show()

## Part 1 (coding task): The nearest identified moth

The simplest rule gives every moth the species of the identified moth nearest to it. Write `nearest_label(X, labeled, y_labeled)`, which returns an array with one label for every row of `X`: for each row, find the closest row among `X[labeled]` and copy its label from `y_labeled`. A matrix of squared distances between all rows and the labeled rows, built with broadcasting as `((X[:, None, :] - X[labeled][None, :, :]) ** 2).sum(axis=-1)`, and `argmin` along axis 1 do it without a loop. The labeled rows get their own labels back, since each is at distance 0 from itself.

This is the baseline. Six labels draw straight boundaries between the identified moths, and the crescents are not straight.

In [ ]:
def nearest_label(X, labeled, y_labeled):
    """The label of the nearest labeled row, for every row of X."""
    # TODO
    raise NotImplementedError

In [ ]:
P = np.array([[0.0, 0.0], [10.0, 0.0], [1.0, 0.0], [9.0, 0.0], [4.0, 0.0]])
assert np.array_equal(nearest_label(P, np.array([0, 1]), np.array([0, 1])), [0, 1, 0, 1, 0]), "(1, 0) and (4, 0) are nearer to (0, 0), and (9, 0) is nearer to (10, 0)"

baseline = nearest_label(X, labeled, y_labeled)
assert baseline.shape == (1000,), f"one label per moth, got shape {baseline.shape}"
assert np.array_equal(baseline[labeled], y_labeled), "an identified moth keeps its own label"
acc_baseline = accuracy(baseline)
print(f"nearest identified moth: {acc_baseline:.3f} of the unlabeled moths right")
assert 0.5 < acc_baseline < 0.97, f"six labels and straight boundaries should land well short of perfect, got {acc_baseline:.3f}"

fig, ax = plt.subplots(figsize=(5, 3.5))
show(ax, baseline, f"nearest identified moth: {acc_baseline:.3f}")
plt.show()
print("Part 1 passed")

## Part 2 (coding task): The most confident guesses

Self-training trusts the classifier's confident predictions. With the nearest-neighbor rule the natural confidence is closeness: a moth right next to an identified one almost surely shares its species, and a moth far from every identified one is a guess. Write `confident_guesses(X, labeled, y_labeled, radius)`, which returns `(rows, guesses)`: the indices of the unlabeled rows whose nearest labeled row is within `radius`, and the labels those rows would copy. Unlabeled means not in `labeled`, and `np.setdiff1d(np.arange(len(X)), labeled)` lists those rows. Compare squared distances with `radius ** 2` to stay clear of square roots.

In [ ]:
def confident_guesses(X, labeled, y_labeled, radius):
    """(rows, guesses): the unlabeled rows within `radius` of a labeled row, and the labels they copy."""
    # TODO
    raise NotImplementedError

In [ ]:
rows, guesses = confident_guesses(P, np.array([0, 1]), np.array([0, 1]), 2.0)
assert np.array_equal(rows, [2, 3]) and np.array_equal(guesses, [0, 1]), "within 2 of an identified point: (1, 0) copies 0 and (9, 0) copies 1, and (4, 0) is too far"
rows, guesses = confident_guesses(P, np.array([0, 1]), np.array([0, 1]), 0.5)
assert len(rows) == 0, "nothing is within 0.5"

rows, guesses = confident_guesses(X, labeled, y_labeled, 0.1)
print(f"within 0.1 of an identified moth: {len(rows)} moths, {np.mean(guesses == y_true[rows]):.3f} of the guesses right")
assert 0 < len(rows) < 200, f"a radius of 0.1 should take a few dozen moths, got {len(rows)}"
assert not np.isin(rows, labeled).any(), "identified moths are never guessed"
assert np.mean(guesses == y_true[rows]) > 0.95, "guesses this close should nearly all be right"
rows_all, _ = confident_guesses(X, labeled, y_labeled, 100.0)
assert len(rows_all) == len(unlabeled), "a huge radius takes every unlabeled moth"
print("Part 2 passed")

## Part 3 (coding task): Absorb, retrain, repeat

Now the loop. Write `self_train(X, labeled, y_labeled, radius)`, which starts from the identified moths and, each round, takes the confident guesses, adds those rows to the labeled set with their guessed labels, and repeats until a round absorbs nothing. Return `(labels, rounds)`: an array with a label for every row, and the number of rounds that absorbed something. A moth never absorbed, because nothing came within `radius` of it, takes the label of its nearest labeled row at the end, which `nearest_label` on the grown labeled set gives for every row at once.

The check runs it with three radii. A small one creeps along each crescent, a few moths at a time. A huge one absorbs everything in the first round, which is the baseline rule again. One in between is big enough to jump the gap between the crescents, and a label that crosses it spreads from there. That is the lesson's confirmation bias: once a pseudo-label is in the training set it looks like truth.

In [ ]:
def self_train(X, labeled, y_labeled, radius):
    """(labels, rounds): every row labeled by rounds of absorbing confident guesses."""
    # TODO
    raise NotImplementedError

In [ ]:
results = {}
for radius in (0.1, 0.4, 5.0):
    labels, rounds = self_train(X, labeled, y_labeled, radius)
    results[radius] = labels
    assert labels.shape == (1000,) and np.array_equal(labels[labeled], y_labeled), "every moth labeled, and the identified ones unchanged"
    print(f"radius {radius}: {rounds:3d} rounds, {accuracy(labels):.3f} right")
assert accuracy(results[0.1]) > 0.98, "a small radius should creep along the crescents and get nearly everything right"
assert np.array_equal(results[5.0], baseline), "a radius that covers everything absorbs it all in round 1, which is the baseline rule"
assert accuracy(results[0.4]) < accuracy(results[0.1]) - 0.02, "a radius that jumps the gap lets labels cross to the other crescent"

fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
for ax, radius in zip(axes, (0.1, 0.4, 5.0)):
    show(ax, results[radius], f"radius {radius}: {accuracy(results[radius]):.3f}")
plt.show()
print("Part 3 passed")

## Part 4 (coding task): Label propagation on five points

Label propagation trusts the geometry directly. Put the labels in a matrix $F$ with one row per point and one column per class, one-hot for the labeled points and zero elsewhere, and let $T$ be the weight matrix with each row divided by its sum. Then repeat two steps: $F \leftarrow TF$, so every point takes the weighted average of its neighbors' rows, and reset the labeled rows to their one-hot labels. Write `propagate(T, Y, labeled)`, which starts from $F = Y$, runs that until the largest change in a sweep is below $10^{-8}$ or for 5,000 sweeps, and returns $F$.

The lesson's example is the path $1 - 2 - 3 - 4 - 5$ with a shortcut $2 - 4$, point 1 labeled $A$ and point 5 labeled $B$. The probability of $A$ at points 2, 3 and 4 comes out as $5/8$, $1/2$ and $3/8$.

In [ ]:
def propagate(T, Y, labeled, tol=1e-8, max_sweeps=5000):
    """F after iterating F <- T F with the labeled rows clamped to Y."""
    # TODO
    raise NotImplementedError

In [ ]:
W5 = np.zeros((5, 5))
for i, j in [(0, 1), (1, 2), (2, 3), (3, 4), (1, 3)]:          # the path 1-2-3-4-5 and the shortcut 2-4
    W5[i, j] = W5[j, i] = 1.0
T5 = W5 / W5.sum(axis=1, keepdims=True)
Y5 = np.zeros((5, 2))
Y5[0, 0] = Y5[4, 1] = 1.0                                       # point 1 is A, point 5 is B
F5 = propagate(T5, Y5, np.array([0, 4]))
print("P(A) at the five points:", F5[:, 0])
assert F5.shape == (5, 2), f"five points and two classes give a 5 x 2 matrix, got {F5.shape}"
assert np.allclose(F5[[0, 4]], Y5[[0, 4]]), "the labeled rows stay clamped to their one-hot labels"
assert np.allclose(F5[:, 0], [1, 5 / 8, 1 / 2, 3 / 8, 0], atol=1e-6), f"the lesson's fixed point is P(A) = 1, 5/8, 1/2, 3/8, 0, got {F5[:, 0]}"
assert np.allclose(F5.sum(axis=1), 1), "each row is a probability distribution over the two classes"
print("Part 4 passed")

## Part 5 (coding task): A similarity graph over the moths

The graph for the moths connects every pair with an RBF weight, $w_{ij} = \exp(-\gamma \lVert \mathbf{x}_i - \mathbf{x}_j \rVert^2)$, so nearby moths are strongly connected and distant ones barely at all. Write `transition_matrix(X, gamma)`, which builds the weight matrix for all rows of `X` and divides each row by its sum, so that row $i$ holds the fraction of moth $i$'s total similarity going to each moth, itself included.

The check propagates the six labels over it with $\gamma = 20$ and scores the result against the hidden species. $Y$ for the moths has 1,000 rows, one-hot in the six identified rows and zero elsewhere, and `argmax` along axis 1 reads the label off each row of $F$.

In [ ]:
def transition_matrix(X, gamma):
    """RBF weights between all rows of X, each row divided by its sum."""
    # TODO
    raise NotImplementedError

In [ ]:
T = transition_matrix(X, 20.0)
assert T.shape == (1000, 1000), "one row and one column per moth"
assert np.allclose(T.sum(axis=1), 1) and T.min() >= 0, "each row holds fractions that sum to 1"
d01 = ((X[0] - X[1]) ** 2).sum()
d02 = ((X[0] - X[2]) ** 2).sum()
assert np.isclose(T[0, 1] / T[0, 2], np.exp(-20.0 * (d01 - d02))), "within a row the entries should be in the ratio of the RBF weights"

Y = np.zeros((1000, 2))
Y[labeled, y_labeled] = 1.0
F = propagate(T, Y, labeled)
propagated = F.argmax(axis=1)
acc_prop = accuracy(propagated)
print(f"label propagation, gamma = 20: {acc_prop:.3f} right")
assert acc_prop > 0.98, "six labels should propagate along the crescents to nearly every moth"

fig, ax = plt.subplots(figsize=(5, 3.5))
show(ax, propagated, f"label propagation: {acc_prop:.3f}")
plt.show()
print("Part 5 passed")

## Part 6 (coding task): The fixed point in one solve

At the fixed point nothing changes, so with the points split into labeled $L$ and unlabeled $U$, $F_U = T_{UU} F_U + T_{UL} Y_L$, a linear system with the solution $F_U = (I - T_{UU})^{-1} T_{UL} Y_L$. Write `propagate_exact(T, Y, labeled)`, which solves it with `np.linalg.solve` and returns the full $F$, with the labeled rows equal to $Y$. `np.ix_(rows, cols)` picks a block out of a matrix. It has to agree with the iteration on the five points and on the moths, and the check finishes with the three methods side by side.

In [ ]:
def propagate_exact(T, Y, labeled):
    """The fixed point of label propagation, by one linear solve."""
    # TODO
    raise NotImplementedError

In [ ]:
F5_exact = propagate_exact(T5, Y5, np.array([0, 4]))
assert F5_exact.shape == (5, 2), "the full F, five rows and two columns"
assert np.allclose(F5_exact[:, 0], [1, 5 / 8, 1 / 2, 3 / 8, 0]), "the solve should land on the same 5/8, 1/2, 3/8"

F_exact = propagate_exact(T, Y, labeled)
assert np.allclose(F_exact[labeled], Y[labeled]), "the labeled rows are their one-hot labels"
assert np.allclose(F_exact, F, atol=1e-3), "the solve and the iteration should agree on the moths"
exact = F_exact.argmax(axis=1)
print(f"closed form: {accuracy(exact):.3f} right, agreeing with the iteration on {np.mean(exact == propagated):.1%} of the moths")
assert np.mean(exact == propagated) > 0.99, "the two routes to the fixed point should give the same labels"

fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
for ax, (name, labels) in zip(axes, [("nearest identified moth", baseline), ("self-training, radius 0.1", results[0.1]), ("label propagation", exact)]):
    show(ax, labels, f"{name}: {accuracy(labels):.3f}")
plt.show()
print("Part 6 passed")

## Solution

### Part 1: The nearest identified moth

In [ ]:
def nearest_label(X, labeled, y_labeled):
    """The label of the nearest labeled row, for every row of X."""
    d_sq = ((X[:, None, :] - X[labeled][None, :, :]) ** 2).sum(axis=-1)
    return y_labeled[d_sq.argmin(axis=1)]

### Part 2: The most confident guesses

In [ ]:
def confident_guesses(X, labeled, y_labeled, radius):
    """(rows, guesses): the unlabeled rows within `radius` of a labeled row, and the labels they copy."""
    rows = np.setdiff1d(np.arange(len(X)), labeled)
    d_sq = ((X[rows][:, None, :] - X[labeled][None, :, :]) ** 2).sum(axis=-1)
    near = d_sq.min(axis=1) <= radius ** 2
    return rows[near], y_labeled[d_sq.argmin(axis=1)][near]

### Part 3: Absorb, retrain, repeat

In [ ]:
def self_train(X, labeled, y_labeled, radius):
    """(labels, rounds): every row labeled by rounds of absorbing confident guesses."""
    labeled, y_labeled = np.array(labeled), np.array(y_labeled)
    rounds = 0
    while True:
        rows, guesses = confident_guesses(X, labeled, y_labeled, radius)
        if len(rows) == 0:
            break
        labeled = np.concatenate([labeled, rows])
        y_labeled = np.concatenate([y_labeled, guesses])
        rounds += 1
    return nearest_label(X, labeled, y_labeled), rounds

### Part 4: Label propagation on five points

In [ ]:
def propagate(T, Y, labeled, tol=1e-8, max_sweeps=5000):
    """F after iterating F <- T F with the labeled rows clamped to Y."""
    F = Y.copy()
    for _ in range(max_sweeps):
        F_next = T @ F
        F_next[labeled] = Y[labeled]
        done = np.abs(F_next - F).max() < tol
        F = F_next
        if done:
            break
    return F

### Part 5: A similarity graph over the moths

In [ ]:
def transition_matrix(X, gamma):
    """RBF weights between all rows of X, each row divided by its sum."""
    d_sq = ((X[:, None, :] - X[None, :, :]) ** 2).sum(axis=-1)
    W = np.exp(-gamma * d_sq)
    return W / W.sum(axis=1, keepdims=True)

### Part 6: The fixed point in one solve

In [ ]:
def propagate_exact(T, Y, labeled):
    """The fixed point of label propagation, by one linear solve."""
    rows = np.setdiff1d(np.arange(len(T)), labeled)
    T_UU = T[np.ix_(rows, rows)]
    T_UL = T[np.ix_(rows, labeled)]
    F = Y.copy()
    F[rows] = np.linalg.solve(np.eye(len(rows)) - T_UU, T_UL @ Y[labeled])
    return F